# Preenchimento de falhas de precipitação mensal

Implementa dois métodos **independentes**, sempre partindo dos dados originais e imutáveis:

1. Regressão linear simples, respeitando o ranking das estações de apoio.
2. Ponderação regional pelo método da razão das médias, usando período comum simultâneo.

Somente blocos de falhas com até o limite configurado (padrão: 6 meses consecutivos) são elegíveis. Valores preenchidos nunca são reutilizados como observações ou apoios.


## Etapa 1 — Configuração

In [13]:

from pathlib import Path
import os
import csv
import json
import math
import hashlib
import shutil
import warnings

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

NOTEBOOK_DIR = Path.cwd()
CONFIG_PATH = Path(os.environ.get(
    "CONFIG_PREENCHIMENTO",
    NOTEBOOK_DIR / "config_preenchimento_falhas.yml"
))

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuração não encontrada: {CONFIG_PATH}\n"
        "Coloque config_preenchimento_falhas.yml na mesma pasta do notebook "
        "ou defina a variável de ambiente CONFIG_PREENCHIMENTO."
    )

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def p_cfg(chave):
    return Path(cfg["paths"][chave]).expanduser()

PATH_PRECIP = p_cfg("precipitacao_mensal")
PATH_APOIO = p_cfg("estacoes_apoio")
OUTPUT_DIR = p_cfg("output_dir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not PATH_PRECIP.exists():
    raise FileNotFoundError(f"Arquivo de precipitação mensal não encontrado: {PATH_PRECIP}")
if not PATH_APOIO.exists():
    raise FileNotFoundError(f"Arquivo de estações de apoio não encontrado: {PATH_APOIO}")

if cfg["controle"].get("sobrescrever_originais", False):
    raise ValueError("controle.sobrescrever_originais deve permanecer false.")

if not cfg["controle"].get("usar_somente_dados_originais", True):
    raise ValueError("Este notebook exige controle.usar_somente_dados_originais = true.")

if cfg["controle"].get("permitir_preenchido_como_apoio", False):
    raise ValueError("Este notebook não permite usar valores preenchidos como apoio.")

trat_neg = str(cfg["regressao_linear"]["tratamento_valor_negativo"]).lower()
if trat_neg not in {"zero", "nan"}:
    raise ValueError("regressao_linear.tratamento_valor_negativo deve ser 'zero' ou 'nan'.")

print("Configuração:", CONFIG_PATH)
print("Precipitação:", PATH_PRECIP)
print("Apoios:", PATH_APOIO)
print("Saída:", OUTPUT_DIR)


Configuração: a:\Codigo\UNIPLU-BR\2. Analise_basin\config_preenchimento_falhas.yml
Precipitação: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\precipitacao_mensal.csv
Apoios: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\estacoes_apoio.csv
Saída: A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas


## Etapa 2 — Carregamento

In [14]:

SEP = cfg["dados"]["separador"]
DECIMAL = cfg["dados"].get("decimal", ".")
ENCODING = cfg["dados"].get("encoding", "utf-8-sig")
COL_DATA = cfg["dados"]["coluna_data"]
FMT_DATA = cfg["dados"]["formato_data"]

def ler_cabecalho_bruto(path, sep, encoding):
    with open(path, "r", encoding=encoding, newline="") as f:
        primeira = f.readline().rstrip("\r\n")
    return next(csv.reader([primeira], delimiter=sep))

cabecalho_precip = ler_cabecalho_bruto(PATH_PRECIP, SEP, ENCODING)
duplicadas_cabecalho = sorted({
    c for c in cabecalho_precip if cabecalho_precip.count(c) > 1
})

precip_raw = pd.read_csv(
    PATH_PRECIP,
    sep=SEP,
    decimal=DECIMAL,
    encoding=ENCODING,
    dtype={COL_DATA: "string"},
    na_values=["NaN", "nan", "NA", "N/A", "null", "None", ""],
    keep_default_na=True,
)

if COL_DATA not in precip_raw.columns:
    raise ValueError(f"Coluna temporal '{COL_DATA}' não encontrada.")

# Nomes das estações permanecem strings exatamente como no cabeçalho.
precip_raw.columns = [str(c) for c in precip_raw.columns]

datas_parse = pd.to_datetime(
    precip_raw[COL_DATA].astype("string"),
    format=FMT_DATA,
    errors="coerce",
)

apoio = pd.read_csv(
    PATH_APOIO,
    sep=SEP,
    decimal=DECIMAL,
    encoding=ENCODING,
    dtype={
        "estacao_alvo": "string",
        "estacao_apoio": "string",
    }
)

colunas_apoio_obrigatorias = {
    "estacao_alvo", "ranking", "estacao_apoio",
    "correlacao_r", "r2", "meses_coincidentes"
}
faltantes_apoio = sorted(colunas_apoio_obrigatorias - set(apoio.columns))
if faltantes_apoio:
    raise ValueError(f"Colunas ausentes em estacoes_apoio.csv: {faltantes_apoio}")

apoio["estacao_alvo"] = apoio["estacao_alvo"].astype("string").str.strip()
apoio["estacao_apoio"] = apoio["estacao_apoio"].astype("string").str.strip()
apoio["ranking"] = pd.to_numeric(apoio["ranking"], errors="coerce")
for c in ["correlacao_r", "r2", "meses_coincidentes"]:
    apoio[c] = pd.to_numeric(apoio[c], errors="coerce")

# Converte as colunas pluviométricas explicitamente para numérico.
estacoes_precip = [c for c in precip_raw.columns if c != COL_DATA]
erros_nao_numericos = {}
for c in estacoes_precip:
    original_txt = precip_raw[c]
    convertido = pd.to_numeric(original_txt, errors="coerce")
    mask_erro = original_txt.notna() & convertido.isna()
    if mask_erro.any():
        erros_nao_numericos[c] = original_txt.loc[mask_erro].astype(str).head(20).tolist()
    precip_raw[c] = convertido

precip = precip_raw.drop(columns=[COL_DATA]).copy()
precip.index = datas_parse
precip.index.name = COL_DATA

print(f"Linhas mensais carregadas: {len(precip):,}")
print(f"Estações pluviométricas: {len(estacoes_precip):,}")
print(f"Linhas de ranking: {len(apoio):,}")
display(apoio.head())


Linhas mensais carregadas: 540
Estações pluviométricas: 19
Linhas de ranking: 146


,estacao_alvo,ranking,estacao_apoio,correlacao_r,r2,meses_coincidentes
0,2751007,1,2751017,0.900560,0.811008,507
1,2751007,2,2852046,0.898935,0.808084,488
2,2751007,3,2851043,0.878377,0.771546,521
3,2751007,4,2851020,0.842247,0.709379,491
4,2751007,5,83914,0.818495,0.669934,449


## Etapa 3 — Validação

In [15]:

def montar_validacao():
    rel = []

    def add(item, status, detalhe, critico=False):
        rel.append({
            "verificacao": item,
            "status": status,
            "critico": bool(critico),
            "detalhe": detalhe,
        })

    # Datas inválidas
    n_datas_invalidas = int(datas_parse.isna().sum())
    add(
        "datas_invalidas",
        "OK" if n_datas_invalidas == 0 else "ERRO",
        f"{n_datas_invalidas} datas não puderam ser convertidas.",
        n_datas_invalidas > 0
    )

    idx_valid = precip.index.dropna()

    # Datas duplicadas
    datas_dup = idx_valid[idx_valid.duplicated()].unique()
    add(
        "datas_duplicadas",
        "OK" if len(datas_dup) == 0 else "ERRO",
        "Nenhuma." if len(datas_dup) == 0 else ", ".join(map(str, datas_dup[:20])),
        len(datas_dup) > 0
    )

    # Ordem temporal
    ordem_ok = idx_valid.is_monotonic_increasing
    add(
        "ordem_temporal",
        "OK" if ordem_ok else "ALERTA",
        "Série já está ordenada." if ordem_ok else "A série será ordenada após a validação.",
        False
    )

    # Colunas duplicadas no cabeçalho bruto
    add(
        "colunas_duplicadas",
        "OK" if not duplicadas_cabecalho else "ERRO",
        "Nenhuma." if not duplicadas_cabecalho else str(duplicadas_cabecalho),
        bool(duplicadas_cabecalho)
    )

    # Dados não numéricos
    add(
        "dados_nao_numericos",
        "OK" if not erros_nao_numericos else "ERRO",
        "Nenhum." if not erros_nao_numericos else json.dumps(erros_nao_numericos, ensure_ascii=False),
        bool(erros_nao_numericos)
    )

    # Ranking inteiro/positivo
    rank_nan = apoio["ranking"].isna()
    rank_nao_int = apoio["ranking"].notna() & (apoio["ranking"] % 1 != 0)
    rank_nao_pos = apoio["ranking"].notna() & (apoio["ranking"] < 1)
    rank_invalido = rank_nan | rank_nao_int | rank_nao_pos
    add(
        "rankings_validos",
        "OK" if not rank_invalido.any() else "ERRO",
        f"{int(rank_invalido.sum())} rankings inválidos.",
        bool(rank_invalido.any())
    )

    # Duplicidade alvo/ranking
    dup_rank = apoio.duplicated(["estacao_alvo", "ranking"], keep=False)
    add(
        "ranking_duplicado_por_alvo",
        "OK" if not dup_rank.any() else "ERRO",
        "Nenhum." if not dup_rank.any() else apoio.loc[dup_rank, ["estacao_alvo", "ranking"]].to_dict("records")[:30],
        bool(dup_rank.any())
    )

    # Duplicidade alvo/apoio
    dup_par = apoio.duplicated(["estacao_alvo", "estacao_apoio"], keep=False)
    add(
        "pares_alvo_apoio_duplicados",
        "OK" if not dup_par.any() else "ERRO",
        "Nenhum." if not dup_par.any() else apoio.loc[dup_par, ["estacao_alvo", "estacao_apoio"]].to_dict("records")[:30],
        bool(dup_par.any())
    )

    # Alvo igual a apoio
    autoapoio = apoio["estacao_alvo"] == apoio["estacao_apoio"]
    add(
        "alvo_igual_apoio",
        "OK" if not autoapoio.any() else "ERRO",
        f"{int(autoapoio.sum())} pares alvo=apoio.",
        bool(autoapoio.any())
    )

    est_set = set(estacoes_precip)
    alvos_inexist = sorted(set(apoio["estacao_alvo"].dropna().astype(str)) - est_set)
    apoios_inexist = sorted(set(apoio["estacao_apoio"].dropna().astype(str)) - est_set)
    add(
        "estacoes_alvo_inexistentes",
        "OK" if not alvos_inexist else "ERRO",
        "Nenhuma." if not alvos_inexist else str(alvos_inexist),
        bool(alvos_inexist)
    )
    add(
        "estacoes_apoio_inexistentes",
        "OK" if not apoios_inexist else "ERRO",
        "Nenhuma." if not apoios_inexist else str(apoios_inexist),
        bool(apoios_inexist)
    )

    # Alvos sem apoios
    cont_apoio = apoio.groupby("estacao_alvo")["estacao_apoio"].count()
    alvos_sem_apoio = cont_apoio[cont_apoio == 0].index.astype(str).tolist()
    add(
        "alvos_sem_estacoes_apoio",
        "OK" if not alvos_sem_apoio else "ERRO",
        "Nenhum." if not alvos_sem_apoio else str(alvos_sem_apoio),
        bool(alvos_sem_apoio)
    )

    # Valores negativos originais
    n_neg = int((precip < 0).sum().sum())
    add(
        "valores_negativos_originais",
        "OK" if n_neg == 0 else "ERRO",
        f"{n_neg} valores negativos.",
        bool(n_neg > 0 and cfg["controle"].get("interromper_em_valores_negativos_originais", True))
    )

    # Continuidade mensal
    if len(idx_valid) > 0:
        periodos = idx_valid.to_period("M")
        esperado = pd.period_range(periodos.min(), periodos.max(), freq="M")
        faltantes = esperado.difference(pd.PeriodIndex(periodos))
    else:
        faltantes = pd.PeriodIndex([], freq="M")
    add(
        "meses_ausentes_na_serie",
        "OK" if len(faltantes) == 0 else "ERRO",
        "Nenhum." if len(faltantes) == 0 else ", ".join(map(str, faltantes[:60])),
        bool(len(faltantes) > 0 and cfg["controle"].get("interromper_em_datas_ausentes", True))
    )

    return pd.DataFrame(rel), faltantes

validacao_df, meses_ausentes = montar_validacao()
display(validacao_df)

erros_criticos = validacao_df[
    (validacao_df["status"] == "ERRO") & (validacao_df["critico"])
]
if not erros_criticos.empty:
    raise ValueError(
        "Validação estrutural falhou. Corrija os itens críticos antes do preenchimento:\n"
        + erros_criticos[["verificacao", "detalhe"]].to_string(index=False)
    )

# Só depois de validar, ordenar e normalizar cada data para início do mês.
precip = precip.sort_index()
precip.index = precip.index.to_period("M").to_timestamp(how="start")
precip.index.name = COL_DATA

apoio["ranking"] = apoio["ranking"].astype(int)
apoio = apoio.sort_values(["estacao_alvo", "ranking"], kind="stable").reset_index(drop=True)

print("Validação concluída.")


,verificacao,status,critico,detalhe
0,datas_invalidas,OK,False,0 datas não puderam ser convertidas.
1,datas_duplicadas,OK,False,Nenhuma.
2,ordem_temporal,OK,False,Série já está ordenada.
3,colunas_duplicadas,OK,False,Nenhuma.
4,dados_nao_numericos,OK,False,Nenhum.
5,rankings_validos,OK,False,0 rankings inválidos.
6,ranking_duplicado_por_alvo,OK,False,Nenhum.
7,pares_alvo_apoio_duplicados,OK,False,Nenhum.
8,alvo_igual_apoio,OK,False,0 pares alvo=apoio.
9,estacoes_alvo_inexistentes,OK,False,Nenhuma.


Validação concluída.


## Etapa 4 — Dados originais imutáveis

In [16]:

# Fonte única de verdade para TODOS os cálculos estatísticos e verificações de disponibilidade.
dados_originais = precip.copy(deep=True)
dados_regressao = dados_originais.copy(deep=True)
dados_ponderacao = dados_originais.copy(deep=True)

# Assinatura de auditoria: deve permanecer idêntica até o final.
hash_dados_originais_inicial = pd.util.hash_pandas_object(
    dados_originais, index=True
).values.tobytes()
hash_dados_originais_inicial = hashlib.sha256(hash_dados_originais_inicial).hexdigest()

alvos = [str(x) for x in apoio["estacao_alvo"].drop_duplicates().tolist()]
ranking_por_alvo = {
    alvo: grp.sort_values("ranking", kind="stable").reset_index(drop=True)
    for alvo, grp in apoio.groupby("estacao_alvo", sort=False)
}

print(f"Estações alvo: {len(alvos)}")
print("dados_originais não será modificado.")


Estações alvo: 19
dados_originais não será modificado.


## Etapa 5 — Identificação dos blocos consecutivos de falha

In [17]:

MAX_MESES = int(cfg["falhas"]["max_meses_consecutivos"])

def detectar_blocos_nan(serie, max_meses=MAX_MESES):
    """Retorna uma linha para cada NaN, informando o bloco consecutivo ao qual pertence."""
    s = serie.sort_index()
    isna = s.isna()

    registros = []
    bloco_id = 0
    i = 0
    n = len(s)

    while i < n:
        if not isna.iloc[i]:
            i += 1
            continue

        ini = i
        while i + 1 < n and isna.iloc[i + 1]:
            # A continuidade cronológica já foi garantida pela validação mensal.
            i += 1
        fim = i
        bloco_id += 1
        tamanho = fim - ini + 1

        for j in range(ini, fim + 1):
            registros.append({
                "data": s.index[j],
                "bloco_id": bloco_id,
                "tamanho_bloco_falha": tamanho,
                "bloco_preenchivel": tamanho <= max_meses,
            })
        i += 1

    return pd.DataFrame(registros)

blocos_por_estacao = {
    est: detectar_blocos_nan(dados_originais[est], MAX_MESES)
    for est in dados_originais.columns
}

n_nan = int(dados_originais.isna().sum().sum())
n_nan_ate = int(sum(
    b["bloco_preenchivel"].sum() if not b.empty else 0
    for b in blocos_por_estacao.values()
))
print(f"NaN originais: {n_nan:,}")
print(f"NaN em blocos de até {MAX_MESES} meses: {n_nan_ate:,}")


NaN originais: 1,063
NaN em blocos de até 6 meses: 408


## Etapa 6 — Regressão linear

In [18]:

MIN_REG = int(cfg["regressao_linear"]["min_observacoes_coincidentes"])
USAR_INTERCEPTO = bool(cfg["regressao_linear"].get("usar_intercepto", True))
TRAT_NEG = str(cfg["regressao_linear"]["tratamento_valor_negativo"]).lower()

def ajustar_regressao_original(alvo, est_apoio):
    pares = dados_originais[[alvo, est_apoio]].dropna()
    n = len(pares)

    if n < MIN_REG:
        return {"valido": False, "motivo": "observacoes_insuficientes", "n": n}

    x = pares[est_apoio].to_numpy(dtype=float)
    y = pares[alvo].to_numpy(dtype=float)

    if not np.isfinite(x).all() or not np.isfinite(y).all():
        return {"valido": False, "motivo": "pares_nao_finitos", "n": n}

    if np.isclose(np.var(x), 0.0):
        return {"valido": False, "motivo": "variancia_apoio_zero", "n": n}

    if USAR_INTERCEPTO:
        b, a = np.polyfit(x, y, 1)
    else:
        denom = float(np.dot(x, x))
        if np.isclose(denom, 0.0):
            return {"valido": False, "motivo": "denominador_zero", "n": n}
        b = float(np.dot(x, y) / denom)
        a = 0.0

    yhat = a + b * x
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if np.isclose(ss_tot, 0.0) else 1.0 - ss_res / ss_tot

    return {
        "valido": True,
        "motivo": "ok",
        "n": int(n),
        "coeficiente_angular": float(b),
        "intercepto": float(a),
        "r2_regressao": float(r2) if np.isfinite(r2) else np.nan,
    }

# Cache seguro: parâmetros dependem somente de dados_originais e do par alvo/apoio.
_cache_reg = {}

def obter_ajuste_reg(alvo, est_apoio):
    chave = (alvo, est_apoio)
    if chave not in _cache_reg:
        _cache_reg[chave] = ajustar_regressao_original(alvo, est_apoio)
    return _cache_reg[chave]

def preencher_regressao():
    resultado = dados_originais.copy(deep=True)
    diag = []

    if not cfg["regressao_linear"].get("enabled", True):
        return resultado, pd.DataFrame(diag)

    for alvo in alvos:
        blocos = blocos_por_estacao[alvo]
        if blocos.empty:
            continue

        rankings = ranking_por_alvo[alvo]

        for _, falha in blocos.iterrows():
            data = falha["data"]
            tam = int(falha["tamanho_bloco_falha"])
            preenchivel = bool(falha["bloco_preenchivel"])

            base = {
                "estacao_alvo": alvo,
                "data": data,
                "tamanho_bloco_falha": tam,
                "bloco_preenchivel": preenchivel,
                "status": "nao_preenchido",
                "motivo": None,
                "estacao_apoio_utilizada": None,
                "ranking_utilizado": np.nan,
                "correlacao_r_ranking": np.nan,
                "r2_ranking": np.nan,
                "meses_coincidentes_ranking": np.nan,
                "n_observacoes_regressao": np.nan,
                "coeficiente_angular": np.nan,
                "intercepto": np.nan,
                "r2_regressao": np.nan,
                "valor_apoio_data_falha": np.nan,
                "valor_estimado": np.nan,
                "tratamento_valor_negativo": TRAT_NEG,
                "previsao_bruta": np.nan,
                "tentativas_rankings": "[]",
            }

            if not preenchivel:
                base["motivo"] = f"bloco_maior_{MAX_MESES}_meses"
                diag.append(base)
                continue

            tentativas = []
            selecionado = False
            houve_disponivel = False
            houve_insuf = False

            for _, rr in rankings.iterrows():
                est_apoio = str(rr["estacao_apoio"])
                rank = int(rr["ranking"])
                valor_apoio = dados_originais.at[data, est_apoio]

                tentativa = {
                    "ranking": rank,
                    "estacao_apoio": est_apoio,
                    "valor_observado_na_data": None if pd.isna(valor_apoio) else float(valor_apoio),
                    "valido": False,
                    "motivo": None,
                }

                if pd.isna(valor_apoio):
                    tentativa["motivo"] = "apoio_nan_na_data"
                    tentativas.append(tentativa)
                    continue

                houve_disponivel = True
                ajuste = obter_ajuste_reg(alvo, est_apoio)

                if not ajuste["valido"]:
                    tentativa["motivo"] = ajuste["motivo"]
                    tentativa["n"] = ajuste.get("n")
                    if ajuste["motivo"] == "observacoes_insuficientes":
                        houve_insuf = True
                    tentativas.append(tentativa)
                    continue

                bruto = float(
                    ajuste["intercepto"]
                    + ajuste["coeficiente_angular"] * float(valor_apoio)
                )

                if not np.isfinite(bruto):
                    tentativa["motivo"] = "previsao_nao_finita"
                    tentativas.append(tentativa)
                    continue

                estimado = bruto
                motivo = f"preenchido_ranking_{rank}"

                if estimado < 0:
                    if TRAT_NEG == "zero":
                        estimado = 0.0
                        motivo = f"preenchido_ranking_{rank}_previsao_negativa_convertida_zero"
                    elif TRAT_NEG == "nan":
                        tentativa["motivo"] = "previsao_negativa"
                        tentativas.append(tentativa)
                        # Continua para o próximo ranking, pois este apoio não permite preencher
                        # sob a regra escolhida.
                        continue

                tentativa["valido"] = True
                tentativa["motivo"] = "selecionado"
                tentativas.append(tentativa)

                resultado.at[data, alvo] = float(estimado)

                base.update({
                    "status": "preenchido",
                    "motivo": motivo,
                    "estacao_apoio_utilizada": est_apoio,
                    "ranking_utilizado": rank,
                    "correlacao_r_ranking": rr.get("correlacao_r", np.nan),
                    "r2_ranking": rr.get("r2", np.nan),
                    "meses_coincidentes_ranking": rr.get("meses_coincidentes", np.nan),
                    "n_observacoes_regressao": ajuste["n"],
                    "coeficiente_angular": ajuste["coeficiente_angular"],
                    "intercepto": ajuste["intercepto"],
                    "r2_regressao": ajuste["r2_regressao"],
                    "valor_apoio_data_falha": float(valor_apoio),
                    "valor_estimado": float(estimado),
                    "previsao_bruta": bruto,
                    "tentativas_rankings": json.dumps(tentativas, ensure_ascii=False),
                })
                selecionado = True
                break

            if not selecionado:
                if not houve_disponivel:
                    base["motivo"] = "nenhuma_estacao_disponivel"
                elif houve_insuf:
                    base["motivo"] = "observacoes_insuficientes"
                else:
                    base["motivo"] = "nenhuma_estacao_valida"
                base["tentativas_rankings"] = json.dumps(tentativas, ensure_ascii=False)

            diag.append(base)

    return resultado, pd.DataFrame(diag)

dados_regressao, diagnostico_regressao = preencher_regressao()
print("Regressão concluída.")
display(diagnostico_regressao["status"].value_counts(dropna=False).rename("n").to_frame())


Regressão concluída.


,n
status,
nao_preenchido,670
preenchido,393


## Etapa 7 — Ponderação regional

In [19]:

MIN_APOIOS = int(cfg["ponderacao_regional"]["min_estacoes_apoio"])
MIN_COMUM = int(cfg["ponderacao_regional"]["min_meses_periodo_comum"])

if not cfg["ponderacao_regional"].get("usar_todas_estacoes_disponiveis", True):
    raise ValueError("Este procedimento exige usar_todas_estacoes_disponiveis = true.")
if not cfg["ponderacao_regional"].get("exigir_periodo_comum", True):
    raise ValueError("Este procedimento exige exigir_periodo_comum = true.")

_cache_comum = {}

def estatisticas_periodo_comum(alvo, apoios_disp):
    # A ordem do ranking é preservada na chave.
    chave = (alvo, tuple(apoios_disp))
    if chave in _cache_comum:
        return _cache_comum[chave]

    cols = [alvo] + list(apoios_disp)
    grupo_comum = dados_originais[cols].dropna()

    info = {
        "n": int(len(grupo_comum)),
        "primeira_data": grupo_comum.index.min() if len(grupo_comum) else pd.NaT,
        "ultima_data": grupo_comum.index.max() if len(grupo_comum) else pd.NaT,
        "media_alvo": float(grupo_comum[alvo].mean()) if len(grupo_comum) else np.nan,
        "medias_apoio": (
            {a: float(grupo_comum[a].mean()) for a in apoios_disp}
            if len(grupo_comum) else {}
        ),
    }
    _cache_comum[chave] = info
    return info

def preencher_ponderacao():
    resultado = dados_originais.copy(deep=True)
    diag = []

    if not cfg["ponderacao_regional"].get("enabled", True):
        return resultado, pd.DataFrame(diag)

    for alvo in alvos:
        blocos = blocos_por_estacao[alvo]
        if blocos.empty:
            continue

        rankings = ranking_por_alvo[alvo]

        for _, falha in blocos.iterrows():
            data = falha["data"]
            tam = int(falha["tamanho_bloco_falha"])
            preenchivel = bool(falha["bloco_preenchivel"])

            base = {
                "estacao_alvo": alvo,
                "data": data,
                "tamanho_bloco_falha": tam,
                "bloco_preenchivel": preenchivel,
                "status": "nao_preenchido",
                "motivo": None,
                "numero_estacoes_disponiveis": 0,
                "estacoes_utilizadas": "",
                "rankings_utilizados": "",
                "n_meses_periodo_comum": np.nan,
                "primeira_data_periodo_comum": pd.NaT,
                "ultima_data_periodo_comum": pd.NaT,
                "media_alvo": np.nan,
                "medias_apoio": "{}",
                "valores_apoio_data_falha": "{}",
                "valor_estimado": np.nan,
            }

            if not preenchivel:
                base["motivo"] = f"bloco_maior_{MAX_MESES}_meses"
                diag.append(base)
                continue

            disp = []
            for _, rr in rankings.iterrows():
                est_apoio = str(rr["estacao_apoio"])
                valor = dados_originais.at[data, est_apoio]
                if pd.notna(valor):
                    disp.append({
                        "estacao": est_apoio,
                        "ranking": int(rr["ranking"]),
                        "valor": float(valor),
                    })

            base["numero_estacoes_disponiveis"] = len(disp)

            if len(disp) < MIN_APOIOS:
                base["motivo"] = "estacoes_disponiveis_insuficientes"
                diag.append(base)
                continue

            apoios_disp = [d["estacao"] for d in disp]
            info = estatisticas_periodo_comum(alvo, apoios_disp)

            base.update({
                "estacoes_utilizadas": "|".join(apoios_disp),
                "rankings_utilizados": "|".join(str(d["ranking"]) for d in disp),
                "n_meses_periodo_comum": info["n"],
                "primeira_data_periodo_comum": info["primeira_data"],
                "ultima_data_periodo_comum": info["ultima_data"],
                "media_alvo": info["media_alvo"],
                "medias_apoio": json.dumps(info["medias_apoio"], ensure_ascii=False),
                "valores_apoio_data_falha": json.dumps(
                    {d["estacao"]: d["valor"] for d in disp},
                    ensure_ascii=False
                ),
            })

            if info["n"] < MIN_COMUM:
                base["motivo"] = "periodo_comum_insuficiente"
                diag.append(base)
                continue

            media_alvo = info["media_alvo"]
            if not np.isfinite(media_alvo) or media_alvo < 0:
                base["motivo"] = "media_alvo_invalida"
                diag.append(base)
                continue

            termos = []
            media_invalida = False
            for d in disp:
                mi = info["medias_apoio"][d["estacao"]]
                # Razão Nx/Ni é indefinida quando Ni=0.
                if not np.isfinite(mi) or mi <= 0:
                    media_invalida = True
                    break
                termos.append((media_alvo / mi) * d["valor"])

            if media_invalida:
                base["motivo"] = "media_apoio_nao_positiva"
                diag.append(base)
                continue

            estimado = float(np.mean(termos))
            if not np.isfinite(estimado):
                base["motivo"] = "estimativa_nao_finita"
                diag.append(base)
                continue
            if estimado < 0:
                # Com originais não negativos e médias positivas isto não deveria ocorrer.
                base["motivo"] = "estimativa_negativa"
                diag.append(base)
                continue

            resultado.at[data, alvo] = estimado
            base.update({
                "status": "preenchido",
                "motivo": "preenchido_ponderacao_regional",
                "valor_estimado": estimado,
            })
            diag.append(base)

    return resultado, pd.DataFrame(diag)

dados_ponderacao, diagnostico_ponderacao = preencher_ponderacao()
print("Ponderação regional concluída.")
display(diagnostico_ponderacao["status"].value_counts(dropna=False).rename("n").to_frame())


Ponderação regional concluída.


,n
status,
nao_preenchido,743
preenchido,320


## Etapa 8 — Diagnósticos e comparação dos métodos

In [20]:

def construir_comparacao():
    rows = []
    for alvo in alvos:
        blocos = blocos_por_estacao[alvo]
        if blocos.empty:
            continue
        for _, f in blocos.iterrows():
            if not bool(f["bloco_preenchivel"]):
                continue
            data = f["data"]
            vr = dados_regressao.at[data, alvo]
            vp = dados_ponderacao.at[data, alvo]

            r_ok = pd.notna(vr)
            p_ok = pd.notna(vp)

            if r_ok and p_ok:
                situacao = "ambos"
                dif = abs(float(vr) - float(vp))
            elif r_ok:
                situacao = "somente_regressao"
                dif = np.nan
            elif p_ok:
                situacao = "somente_ponderacao"
                dif = np.nan
            else:
                situacao = "nenhum"
                dif = np.nan

            rows.append({
                "estacao": alvo,
                "data": data,
                "valor_original": np.nan,
                "regressao": vr,
                "ponderacao_regional": vp,
                "situacao": situacao,
                "diferenca_absoluta": dif,
            })
    return pd.DataFrame(rows)

comparacao_metodos = construir_comparacao()
display(comparacao_metodos["situacao"].value_counts(dropna=False).rename("n").to_frame())


,n
situacao,
ambos,320
somente_regressao,73
nenhum,15


## Etapa 9 — Testes automáticos obrigatórios

In [21]:

testes = []

def registrar_teste(numero, nome, ok, detalhe=""):
    testes.append({
        "teste": numero,
        "nome": nome,
        "resultado": "OK" if ok else "FALHOU",
        "detalhe": detalhe,
    })
    if not ok:
        raise AssertionError(f"Teste {numero} falhou — {nome}: {detalhe}")

mask_obs = dados_originais.notna()

# Teste 1 — observados originais intocados
ok1 = (
    dados_regressao.where(mask_obs).equals(dados_originais.where(mask_obs))
    and dados_ponderacao.where(mask_obs).equals(dados_originais.where(mask_obs))
)
registrar_teste(1, "Nenhum valor originalmente observado foi modificado", ok1)

# Teste 2 — blocos > limite não preenchidos
ok2 = True
for est, b in blocos_por_estacao.items():
    if b.empty:
        continue
    for _, f in b.loc[~b["bloco_preenchivel"]].iterrows():
        d = f["data"]
        if pd.notna(dados_regressao.at[d, est]) or pd.notna(dados_ponderacao.at[d, est]):
            ok2 = False
            break
    if not ok2:
        break
registrar_teste(2, f"Nenhum valor de bloco > {MAX_MESES} meses foi preenchido", ok2)

# Teste 3 — apoios vieram exclusivamente do original
ok3 = True
if not diagnostico_regressao.empty:
    for _, r in diagnostico_regressao.loc[diagnostico_regressao["status"] == "preenchido"].iterrows():
        val_orig = dados_originais.at[r["data"], str(r["estacao_apoio_utilizada"])]
        if pd.isna(val_orig) or not np.isclose(float(val_orig), float(r["valor_apoio_data_falha"])):
            ok3 = False
            break
if ok3 and not diagnostico_ponderacao.empty:
    for _, r in diagnostico_ponderacao.loc[diagnostico_ponderacao["status"] == "preenchido"].iterrows():
        vals = json.loads(r["valores_apoio_data_falha"])
        for est, v in vals.items():
            vo = dados_originais.at[r["data"], est]
            if pd.isna(vo) or not np.isclose(float(vo), float(v)):
                ok3 = False
                break
        if not ok3:
            break
registrar_teste(3, "Nenhum valor estimado foi reutilizado como apoio", ok3)

# Testes 4 e 5 — ordem do ranking e validade das anteriores
ok4 = True
ok5 = True
for _, r in diagnostico_regressao.loc[diagnostico_regressao["status"] == "preenchido"].iterrows():
    tent = json.loads(r["tentativas_rankings"])
    ranks = [int(t["ranking"]) for t in tent]
    if ranks != sorted(ranks):
        ok4 = False
        break
    sel = int(r["ranking_utilizado"])
    anteriores = [t for t in tent if int(t["ranking"]) < sel]
    if any(bool(t.get("valido")) for t in anteriores):
        ok5 = False
        break
    selecionados = [t for t in tent if t.get("motivo") == "selecionado"]
    if len(selecionados) != 1 or int(selecionados[0]["ranking"]) != sel:
        ok5 = False
        break
registrar_teste(4, "Regressão respeitou a ordem crescente do ranking", ok4)
registrar_teste(5, "Ranking inferior só foi usado após anteriores inválidos/indisponíveis", ok5)

# Teste 6 — ponderação nunca usa menos que o mínimo
pre_p = diagnostico_ponderacao.loc[diagnostico_ponderacao["status"] == "preenchido"]
ok6 = bool((pre_p["numero_estacoes_disponiveis"] >= MIN_APOIOS).all()) if len(pre_p) else True
registrar_teste(6, f"Ponderação utilizou pelo menos {MIN_APOIOS} estações", ok6)

# Teste 7 — mesmo período comum para alvo e todos os apoios
ok7 = True
for _, r in pre_p.iterrows():
    ests = [x for x in str(r["estacoes_utilizadas"]).split("|") if x]
    grupo = dados_originais[[r["estacao_alvo"]] + ests].dropna()
    if len(grupo) != int(r["n_meses_periodo_comum"]):
        ok7 = False
        break
    if len(grupo):
        if pd.Timestamp(r["primeira_data_periodo_comum"]) != grupo.index.min():
            ok7 = False
            break
        if pd.Timestamp(r["ultima_data_periodo_comum"]) != grupo.index.max():
            ok7 = False
            break
        if not np.isclose(float(r["media_alvo"]), float(grupo[r["estacao_alvo"]].mean())):
            ok7 = False
            break
        medias = json.loads(r["medias_apoio"])
        for e in ests:
            if not np.isclose(float(medias[e]), float(grupo[e].mean())):
                ok7 = False
                break
    if not ok7:
        break
registrar_teste(7, "Médias da ponderação usam exatamente as mesmas datas", ok7)

# Teste 8 — ambos começaram de cópias do mesmo original
ok8 = True
for df in (dados_regressao, dados_ponderacao):
    # Nas posições observadas, devem ser idênticos ao original.
    if not df.where(mask_obs).equals(dados_originais.where(mask_obs)):
        ok8 = False
        break
registrar_teste(8, "Regressão e ponderação começaram dos mesmos dados originais", ok8)

# Teste 9 — falhas estruturalmente não preenchíveis continuam NaN
ok9 = ok2
registrar_teste(9, "Falhas em blocos acima do limite continuam NaN", ok9)

# Teste 10 — apoio nunca estava NaN na data preenchida
ok10 = True
for _, r in diagnostico_regressao.loc[diagnostico_regressao["status"] == "preenchido"].iterrows():
    if pd.isna(dados_originais.at[r["data"], str(r["estacao_apoio_utilizada"])]):
        ok10 = False
        break
if ok10:
    for _, r in pre_p.iterrows():
        for e in str(r["estacoes_utilizadas"]).split("|"):
            if e and pd.isna(dados_originais.at[r["data"], e]):
                ok10 = False
                break
        if not ok10:
            break
registrar_teste(10, "Todo apoio possuía observação original na data preenchida", ok10)

# Teste 11 — preenchidos são finitos
vals_reg = diagnostico_regressao.loc[
    diagnostico_regressao["status"] == "preenchido", "valor_estimado"
].to_numpy(dtype=float)
vals_pon = pre_p["valor_estimado"].to_numpy(dtype=float)
ok11 = np.isfinite(vals_reg).all() and np.isfinite(vals_pon).all()
registrar_teste(11, "Todos os valores preenchidos são finitos", bool(ok11))

# Teste 12 — sem valores negativos no resultado final
ok12 = bool((dados_regressao.dropna() >= 0).all().all() and
            (dados_ponderacao.dropna() >= 0).all().all())
registrar_teste(12, "Não existem valores negativos nos resultados finais", ok12)

# Garantia adicional: dados_originais não foi alterado em memória.
hash_final = pd.util.hash_pandas_object(dados_originais, index=True).values.tobytes()
hash_final = hashlib.sha256(hash_final).hexdigest()
if hash_final != hash_dados_originais_inicial:
    raise AssertionError("dados_originais foi modificado durante o processamento.")

testes_df = pd.DataFrame(testes)
display(testes_df)
print("Todos os testes obrigatórios passaram.")


,teste,nome,resultado,detalhe
0,1,Nenhum valor originalmente observado foi modif...,OK,
1,2,Nenhum valor de bloco > 6 meses foi preenchido,OK,
2,3,Nenhum valor estimado foi reutilizado como apoio,OK,
3,4,Regressão respeitou a ordem crescente do ranking,OK,
4,5,Ranking inferior só foi usado após anteriores ...,OK,
5,6,Ponderação utilizou pelo menos 3 estações,OK,
6,7,Médias da ponderação usam exatamente as mesmas...,OK,
7,8,Regressão e ponderação começaram dos mesmos da...,OK,
8,9,Falhas em blocos acima do limite continuam NaN,OK,
9,10,Todo apoio possuía observação original na data...,OK,


Todos os testes obrigatórios passaram.


## Etapa 10 — Exportação

In [22]:

SAIDA_CFG = cfg["saida"]
REP_NAN = cfg["dados"].get("representacao_nan", "NaN")

def exportar_serie(df, path):
    out = df.copy()
    out.insert(0, COL_DATA, out.index.strftime("%Y-%m"))
    out.to_csv(
        path,
        sep=SEP,
        decimal=DECIMAL,
        encoding=ENCODING,
        index=False,
        na_rep=REP_NAN
    )

def preparar_diag_export(df):
    out = df.copy()
    if "data" in out.columns:
        out["data"] = pd.to_datetime(out["data"]).dt.strftime("%Y-%m")
    for c in ["primeira_data_periodo_comum", "ultima_data_periodo_comum"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.strftime("%Y-%m")
    return out

arquivo_reg = OUTPUT_DIR / SAIDA_CFG["precipitacao_regressao"]
arquivo_pon = OUTPUT_DIR / SAIDA_CFG["precipitacao_ponderacao"]
arquivo_dreg = OUTPUT_DIR / SAIDA_CFG["diagnostico_regressao"]
arquivo_dpon = OUTPUT_DIR / SAIDA_CFG["diagnostico_ponderacao"]
arquivo_comp = OUTPUT_DIR / SAIDA_CFG["comparacao_metodos"]
arquivo_resumo = OUTPUT_DIR / SAIDA_CFG["resumo"]
arquivo_testes = OUTPUT_DIR / SAIDA_CFG.get("testes", "testes_consistencia.csv")
arquivo_cfg = OUTPUT_DIR / SAIDA_CFG.get("config_utilizada", "config_preenchimento_falhas_utilizada.yml")

exportar_serie(dados_regressao, arquivo_reg)
exportar_serie(dados_ponderacao, arquivo_pon)

preparar_diag_export(diagnostico_regressao).to_csv(
    arquivo_dreg, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)
preparar_diag_export(diagnostico_ponderacao).to_csv(
    arquivo_dpon, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

comp_exp = comparacao_metodos.copy()
if not comp_exp.empty:
    comp_exp["data"] = pd.to_datetime(comp_exp["data"]).dt.strftime("%Y-%m")
comp_exp.to_csv(
    arquivo_comp, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

testes_df.to_csv(
    arquivo_testes, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

shutil.copy2(CONFIG_PATH, arquivo_cfg)

print("Arquivos principais exportados:")
for p in [arquivo_reg, arquivo_pon, arquivo_dreg, arquivo_dpon, arquivo_comp, arquivo_testes, arquivo_cfg]:
    print(" -", p)


Arquivos principais exportados:
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\precipitacao_mensal_regressao_linear.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\precipitacao_mensal_ponderacao_regional.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\diagnostico_regressao_linear.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\diagnostico_ponderacao_regional.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\comparacao_metodos.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\preenchimento_falhas\testes_consistencia.csv
 - A:\Codigo\UNIPLU-BR\2. Analise_basin\resultados_consistencia\basin_86500000_2809_131052\pree

## Etapa 11 — Resumo final

In [23]:

def resumo_por_estacao():
    rows = []
    for est in dados_originais.columns:
        orig = dados_originais[est]
        b = blocos_por_estacao[est]

        n_total = len(orig)
        n_obs = int(orig.notna().sum())
        n_nan_orig = int(orig.isna().sum())
        n_ate = int(b["bloco_preenchivel"].sum()) if not b.empty else 0
        n_maior = int((~b["bloco_preenchivel"]).sum()) if not b.empty else 0

        pre_reg = int((orig.isna() & dados_regressao[est].notna()).sum())
        pre_pon = int((orig.isna() & dados_ponderacao[est].notna()).sum())

        rest_reg = int(dados_regressao[est].isna().sum())
        rest_pon = int(dados_ponderacao[est].isna().sum())

        den = n_ate
        pct_reg = 100.0 * pre_reg / den if den else np.nan
        pct_pon = 100.0 * pre_pon / den if den else np.nan

        rows.append({
            "estacao": est,
            "n_total_meses": n_total,
            "n_observados_originais": n_obs,
            "n_nan_originais": n_nan_orig,
            f"n_nan_em_blocos_ate_{MAX_MESES}": n_ate,
            f"n_nan_em_blocos_maiores_{MAX_MESES}": n_maior,
            "n_preenchidos_regressao": pre_reg,
            "n_preenchidos_ponderacao": pre_pon,
            "n_restantes_regressao": rest_reg,
            "n_restantes_ponderacao": rest_pon,
            "percentual_preenchido_regressao": pct_reg,
            "percentual_preenchido_ponderacao": pct_pon,
        })

    res = pd.DataFrame(rows)

    global_row = {
        "estacao": "__GLOBAL__",
        "n_total_meses": int(res["n_total_meses"].sum()),
        "n_observados_originais": int(res["n_observados_originais"].sum()),
        "n_nan_originais": int(res["n_nan_originais"].sum()),
        f"n_nan_em_blocos_ate_{MAX_MESES}": int(res[f"n_nan_em_blocos_ate_{MAX_MESES}"].sum()),
        f"n_nan_em_blocos_maiores_{MAX_MESES}": int(res[f"n_nan_em_blocos_maiores_{MAX_MESES}"].sum()),
        "n_preenchidos_regressao": int(res["n_preenchidos_regressao"].sum()),
        "n_preenchidos_ponderacao": int(res["n_preenchidos_ponderacao"].sum()),
        "n_restantes_regressao": int(res["n_restantes_regressao"].sum()),
        "n_restantes_ponderacao": int(res["n_restantes_ponderacao"].sum()),
    }
    den = global_row[f"n_nan_em_blocos_ate_{MAX_MESES}"]
    global_row["percentual_preenchido_regressao"] = (
        100.0 * global_row["n_preenchidos_regressao"] / den if den else np.nan
    )
    global_row["percentual_preenchido_ponderacao"] = (
        100.0 * global_row["n_preenchidos_ponderacao"] / den if den else np.nan
    )

    return pd.concat([res, pd.DataFrame([global_row])], ignore_index=True)

resumo_preenchimento = resumo_por_estacao()
resumo_preenchimento.to_csv(
    arquivo_resumo, sep=SEP, decimal=DECIMAL, encoding=ENCODING,
    index=False, na_rep=REP_NAN
)

display(resumo_preenchimento.tail(min(10, len(resumo_preenchimento))))

print("\nResumo metodológico:")
print(f"1. Blocos: sequências contínuas de NaN na série mensal; somente tamanho <= {MAX_MESES} é elegível.")
print("2. Regressão: apoios percorridos em ranking crescente; usa o primeiro apoio observado e estatisticamente válido.")
print("3. Ajuste: P_alvo = a + b*P_apoio, estimado exclusivamente com pares observados em dados_originais.")
print("4. Ponderação: usa todas as estações cadastradas como apoio que tenham observação original na data da falha.")
print("5. Período comum: dropna simultâneo em ALVO + todos os apoios usados naquela falha.")
print("6. Valores preenchidos não são reutilizados: toda consulta e todo ajuste usam dados_originais.")
print("7. Antes da exportação são executados 12 testes obrigatórios de consistência.")
print("8. Foram produzidos os dois resultados, dois diagnósticos, comparação, resumo, testes e cópia da configuração.")
print("\nResumo global:")
display(resumo_preenchimento.loc[resumo_preenchimento["estacao"] == "__GLOBAL__"])


,estacao,n_total_meses,n_observados_originais,n_nan_originais,n_nan_em_blocos_ate_6,n_nan_em_blocos_maiores_6,n_preenchidos_regressao,n_preenchidos_ponderacao,n_restantes_regressao,n_restantes_ponderacao,percentual_preenchido_regressao,percentual_preenchido_ponderacao
10,2851044,540,417,123,27,96,27,27,96,96,100.000000,100.000000
11,2852004,540,517,23,23,0,23,23,0,0,100.000000,100.000000
12,2852014,540,499,41,3,38,3,3,38,38,100.000000,100.000000
13,2852016,540,395,145,1,144,1,1,144,144,100.000000,100.000000
14,2852031,540,492,48,9,39,9,9,39,39,100.000000,100.000000
15,2852046,540,493,47,47,0,47,47,0,0,100.000000,100.000000
16,2951010,540,464,76,66,10,56,0,20,76,84.848485,0.000000
17,2951070,540,418,122,36,86,36,33,86,89,100.000000,91.666667
18,83914,540,460,80,37,43,37,37,43,43,100.000000,100.000000
19,__GLOBAL__,10260,9197,1063,408,655,393,320,670,743,96.323529,78.431373



Resumo metodológico:
1. Blocos: sequências contínuas de NaN na série mensal; somente tamanho <= 6 é elegível.
2. Regressão: apoios percorridos em ranking crescente; usa o primeiro apoio observado e estatisticamente válido.
3. Ajuste: P_alvo = a + b*P_apoio, estimado exclusivamente com pares observados em dados_originais.
4. Ponderação: usa todas as estações cadastradas como apoio que tenham observação original na data da falha.
5. Período comum: dropna simultâneo em ALVO + todos os apoios usados naquela falha.
6. Valores preenchidos não são reutilizados: toda consulta e todo ajuste usam dados_originais.
7. Antes da exportação são executados 12 testes obrigatórios de consistência.
8. Foram produzidos os dois resultados, dois diagnósticos, comparação, resumo, testes e cópia da configuração.

Resumo global:


,estacao,n_total_meses,n_observados_originais,n_nan_originais,n_nan_em_blocos_ate_6,n_nan_em_blocos_maiores_6,n_preenchidos_regressao,n_preenchidos_ponderacao,n_restantes_regressao,n_restantes_ponderacao,percentual_preenchido_regressao,percentual_preenchido_ponderacao
19,__GLOBAL__,10260,9197,1063,408,655,393,320,670,743,96.323529,78.431373
